# Exercise Sheet 4 — Principal component analysis
**Course:** Big Data Science in Finance (WI3436TU)
**Data:** same as Sheet 3, Problem 3 — `../Lab_3/E3/market_data_2007_2025.csv`
(SPY, QQQ, TLT, HYG, VIX)

This notebook works through Problems 1-3 of Exercise Sheet 4. Code cells compute the required
quantities and produce the plots; markdown cells contain the hand calculations and written
discussion. Divisor $m$ (number of dates, `ddof=0`) is used throughout.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:0.6f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

MARKET_PATH = Path("../Lab_3/E3/market_data_2007_2025.csv")
ASSETS = ["SPY", "QQQ", "TLT", "HYG", "VIX"]

### Notation

As in Lectures 3–4 and Sheet 3, put financial series in rows and dates in columns. For $n$
series and $m$ dates, let $\mathbf R \in \mathbb R^{n \times m}$, $\bar{\mathbf r} = \mathbf R\mathbf 1_m / m$,
and $\mathbf X = \mathbf R - \bar{\mathbf r}\mathbf 1_m^\top$. For its rank-$r$ SVD,
$$\mathbf X = \mathbf U_r\boldsymbol\Sigma_r\mathbf V_r^\top, \qquad \widehat{\mathbf C} = \frac{1}{m}\mathbf X\mathbf X^\top, \qquad \mathbf Y = \mathbf U_r^\top\mathbf X = \boldsymbol\Sigma_r\mathbf V_r^\top.$$
The principal direction $\mathbf u_k \in \mathbb R^n$ gives the series loadings. Row $k$ of
$\mathbf Y$, written as a column, is the dated score series
$\mathbf y_k = \mathbf X^\top\mathbf u_k = \sigma_k\mathbf v_k$; its variance is $\lambda_k = \sigma_k^2/m$.

For a symmetric covariance or correlation matrix, `np.linalg.eigh` returns eigenvalues in
**increasing** order.

## Problem 1

Return to $\mathbf A$ from Sheet 3, Problem 1(a). Interpret its columns as two dates of returns
for two assets, in percentage points:
$$\mathbf R = \mathbf A = \begin{pmatrix} 3 & 0 \\ 4 & 0 \end{pmatrix}.$$

In [ ]:
A = np.array([[3.0, 0.0],
              [4.0, 0.0]])
m_A = A.shape[1]   # number of dates

### a)

Find $\bar{\mathbf r}$ and the centered matrix $\mathbf X$. Check that each row of $\mathbf X$
sums to zero.

**Hand calculation.**

In [ ]:
r_bar = A.mean(axis=1, keepdims=True)   # row means, shape (2, 1)
X_A = A - r_bar

print("r_bar =", r_bar.flatten())
print("X =")
print(X_A)
print("row sums of X:", X_A.sum(axis=1))

### b)

Compute $\widehat{\mathbf C} = \mathbf X\mathbf X^\top/2$. Find its eigenvalues and corresponding
orthonormal eigenvectors by hand. Choose $\mathbf u_1$ to have positive entries.

**Hand calculation.**

In [ ]:
C_A = X_A @ X_A.T / m_A
eigvals, eigvecs = np.linalg.eigh(C_A)
eigvals, eigvecs = eigvals[::-1], eigvecs[:, ::-1]   # sort in decreasing order

u1 = eigvecs[:, 0]
if u1[0] < 0:          # choose u1 with positive entries
    u1 = -u1
lam1 = eigvals[0]

print("C =")
print(C_A)
print("eigenvalues (decreasing):", eigvals)
print("eigenvectors (columns):")
print(eigvecs)
print("u1 =", u1)

### c)

Use $\sigma_1 = \sqrt{2\lambda_1}$ and $\mathbf v_1 = \mathbf X^\top\mathbf u_1/\sigma_1$ to obtain
$\mathbf X = \sigma_1\mathbf u_1\mathbf v_1^\top$. Compared with the uncentered SVD in Sheet 3,
which of $\mathbf u_1$, $\sigma_1$, and $\mathbf v_1$ have changed?

In [ ]:
sigma1 = np.sqrt(2 * lam1)
v1 = X_A.T @ u1 / sigma1

print(f"sigma1 = {sigma1:.4f}")
print("v1 =", v1)
print("sigma1 * u1 v1^T =")
print(sigma1 * np.outer(u1, v1))

# uncentered SVD of A from Sheet 3, for comparison
U_raw, s_raw, Vt_raw = np.linalg.svd(A, full_matrices=False)
sign = np.sign(U_raw[0, 0])
print("\nSheet 3 (uncentered): u1 =", sign * U_raw[:, 0], " sigma1 =", s_raw[0], " v1 =", sign * Vt_raw[0])

**Discussion.**

### d)

Calculate the two scores in $\mathbf y_1 = \mathbf X^\top\mathbf u_1$. Interpret their signs and
verify directly that their mean is zero and their variance is $\lambda_1$. What fraction of total
variance does PC1 explain?

In [ ]:
y1 = X_A.T @ u1

print("scores y1 =", y1)
print(f"mean of y1     = {y1.mean():.4f}")
print(f"variance of y1 = {y1.var():.4f}   (lambda1 = {lam1:.4f})")   # np.var uses ddof=0
print(f"fraction of total variance explained by PC1 = {lam1 / eigvals.sum():.4f}")

**Discussion.**

### e)

Reconstruct both original return columns using $\mathbf r_t = \bar{\mathbf r} + y_{1t}\mathbf u_1$.

In [ ]:
for t in range(m_A):
    r_t = r_bar.flatten() + y1[t] * u1
    print(f"date {t + 1}: reconstructed r_t = {r_t},  original = {A[:, t]}")

## Problem 2

Continue with the same financial data as Sheet 3, Problem 3. Reuse your uncentered log-return
change matrix $\mathbf R$ and its dates, with rows SPY, QQQ, TLT, HYG, VIX, in that order. Here
$n = 5$ and $m = T$, where $T$ was the number of dates in Sheet 3.

In [ ]:
# same construction as Sheet 3, Problem 3
prices = pd.read_csv(MARKET_PATH, index_col=0, parse_dates=True).sort_index()
prices.columns = [c.replace("^", "") for c in prices.columns]   # e.g. "^VIX" -> "VIX"
prices = prices[ASSETS]

log_changes = pd.concat(
    {a: np.log(prices[a].dropna()).diff().dropna() for a in ASSETS},
    axis=1, join="inner",
)

R = log_changes.T.to_numpy()    # n x m: series in rows, dates in columns
dates = log_changes.index
n, m = R.shape

### a)

Verify that the shape, first and last dates, and row order of $\mathbf R$ agree with Sheet 3.
Create the centered matrix $\mathbf X$ and the standardized matrix
$$\mathbf Z = \mathbf D^{-1}\mathbf X, \qquad \mathbf D = \operatorname{diag}(s_1, \dots, s_5), \qquad s_i^2 = \frac{1}{m}\sum_{t=1}^m x_{it}^2.$$
Verify numerically that the rows of $\mathbf Z$ have mean zero and variance one.

In [ ]:
print("R shape:", R.shape)
print("first date:", dates[0].date(), " last date:", dates[-1].date())
print("row order:", list(log_changes.columns))

X = R - R.mean(axis=1, keepdims=True)        # centered
s_i = np.sqrt(np.mean(X**2, axis=1))          # standard deviations with divisor m
Z = X / s_i[:, None]                          # Z = D^{-1} X

pd.DataFrame({"s_i": s_i, "row mean of Z": Z.mean(axis=1), "row variance of Z": Z.var(axis=1)},
             index=ASSETS)

### b)

What role does centring play before PCA, and what additional modeling choice is made, when
$\mathbf X$ is standardized to $\mathbf Z$?

**Discussion.**

## Problem 3

Use the standardized data matrix $\mathbf Z$ from Problem 2.

### a)

Compute $\mathbf Z = \mathbf U\boldsymbol\Sigma\mathbf V^\top$ and the scores
$\mathbf Y = \mathbf U^\top\mathbf Z$. Also compute $\widehat{\mathbf C}_Z = \mathbf Z\mathbf Z^\top/m$
and its eigenpairs, ordered so that $\lambda_1 \ge \dots \ge \lambda_5$. Orient PC1 so that its
score has a positive same-date correlation with SPY; orient each other component so that its
largest absolute loading is positive. When changing the sign of an SVD singular vector, change
both corresponding singular vectors. Align the covariance eigenvector signs with the SVD
directions and verify
$$\lambda_k = \frac{\sigma_k^2}{m}, \qquad \mathbf u^{(\text{cov})}_k = \mathbf u^{(\text{svd})}_k, \qquad \mathbf Z^\top\mathbf u_k = \sigma_k\mathbf v_k.$$

In [ ]:
# --- SVD route ---
U, sig, Vt = np.linalg.svd(Z, full_matrices=False)

# orientation: PC1 score positively correlated with SPY (row 0 of Z),
# other PCs: largest absolute loading positive. Flip u_k and v_k together.
for k in range(n):
    if k == 0:
        score = sig[0] * Vt[0]
        flip = np.corrcoef(score, Z[0])[0, 1] < 0
    else:
        flip = U[np.argmax(np.abs(U[:, k])), k] < 0
    if flip:
        U[:, k] *= -1
        Vt[k, :] *= -1

Y = U.T @ Z    # scores, one row per PC

# --- covariance (eigen) route ---
C_Z = Z @ Z.T / m
lam, W = np.linalg.eigh(C_Z)
lam, W = lam[::-1], W[:, ::-1]    # decreasing order

# align eigenvector signs with the SVD directions
for k in range(n):
    if W[:, k] @ U[:, k] < 0:
        W[:, k] *= -1

pc_names = [f"PC{k + 1}" for k in range(n)]
print("eigenvalues lambda_k:", np.round(lam, 6))
print("sigma_k^2 / m:       ", np.round(sig**2 / m, 6))
print("max |u_cov - u_svd|        =", np.max(np.abs(W - U)))
print("max |Z^T u_k - sigma_k v_k| =", np.max(np.abs(Z.T @ U - Vt.T * sig)))
pd.DataFrame(U, index=ASSETS, columns=pc_names)

### b)

For equally shaped arrays $\mathbf A$ and $\mathbf B$, define their maximum absolute discrepancy by
$d(\mathbf A, \mathbf B) = \max_{i,j}|A_{ij} - B_{ij}|$; for vectors, take the maximum over their
entries. Compute this discrepancy between the two eigenvalue vectors, the two loading matrices,
and the two score matrices in part a). Explain and verify the following equality:
$\frac{1}{m}\mathbf Z\mathbf Z^\top = \mathbf U\frac{\boldsymbol\Sigma^2}{m}\mathbf U^\top$.

In [ ]:
def d(A, B):
    return np.max(np.abs(A - B))

Y_cov = W.T @ Z    # scores from the covariance eigenvectors

print(f"d(eigenvalues)     = {d(lam, sig**2 / m):.2e}")
print(f"d(loadings)        = {d(W, U):.2e}")
print(f"d(scores)          = {d(Y_cov, Y):.2e}")
print(f"d(ZZ^T/m, U S^2/m U^T) = {d(Z @ Z.T / m, U @ np.diag(sig**2 / m) @ U.T):.2e}")

**Discussion.**

### c)

Compute the following three quantities separately and verify that they agree:
$$\sum_{i=1}^5 \widehat{\operatorname{Var}}(Z_i) = \frac{1}{m}\|\mathbf Z\|_F^2 = \sum_{k=1}^5 \lambda_k.$$
Define the individual and cumulative explained-variance fractions by
$$\mathrm{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^5 \lambda_j}, \qquad \mathrm{CEV}_K = \sum_{k=1}^K \mathrm{EVR}_k.$$
Plot both. Choose the smallest $K$ with $\mathrm{CEV}_K \ge 80\%$, and report the fractions of total
variance preserved and omitted.

In [ ]:
total_var_rows = Z.var(axis=1).sum()
total_var_frob = np.linalg.norm(Z, "fro")**2 / m
total_var_eig = lam.sum()
print(f"sum of row variances = {total_var_rows:.6f}")
print(f"||Z||_F^2 / m        = {total_var_frob:.6f}")
print(f"sum of eigenvalues   = {total_var_eig:.6f}")

EVR = lam / lam.sum()
CEV = np.cumsum(EVR)
K = int(np.argmax(CEV >= 0.80)) + 1

ks = np.arange(1, n + 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(ks, EVR, alpha=0.7, label="EVR_k")
ax.plot(ks, CEV, "o-", color="tab:orange", label="CEV_K")
ax.axhline(0.8, color="gray", ls="--", label="80%")
ax.set_xticks(ks)
ax.set_xlabel("k")
ax.set_title("Explained-variance fractions (correlation PCA)")
ax.legend()
plt.show()

print(f"smallest K with CEV_K >= 80%: K = {K}")
print(f"variance preserved = {CEV[K - 1]:.2%}, omitted = {1 - CEV[K - 1]:.2%}")

### d)

Plot the PC1 and PC2 loadings, score series from above, and interpret them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for k, ax in enumerate(axes):
    ax.bar(ASSETS, U[:, k])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"PC{k + 1} loadings")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for k, ax in enumerate(axes):
    ax.plot(dates, Y[k], lw=0.6)
    ax.set_ylabel(f"PC{k + 1} score")
axes[0].set_title("PC1 and PC2 score series")
fig.tight_layout()
plt.show()

**Discussion.**

### e)

For PC1 and PC2, identify the date with the largest absolute score and explain its sign from the
corresponding column of $\mathbf Z$.

In [ ]:
for k in range(2):
    t = int(np.argmax(np.abs(Y[k])))
    print(f"PC{k + 1}: largest |score| on {dates[t].date()}, score = {Y[k, t]:.3f}")
    print(pd.DataFrame({"Z column": Z[:, t], f"PC{k + 1} loading": U[:, k],
                        "loading * Z": U[:, k] * Z[:, t]}, index=ASSETS))
    print()

**Discussion.**

### f)

Use the same $\mathbf R \in \mathbb R^{5 \times m}$, dates, and row order in all three analyses:

| matrix | analysis |
|---|---|
| $\mathbf R$ | raw SVD from Sheet 3 |
| $\mathbf X = \mathbf R - \bar{\mathbf r}\mathbf 1_m^\top$ | covariance PCA |
| $\mathbf Z = \mathbf D^{-1}\mathbf X$ | correlation PCA. |

For this comparison, orient each leading direction so that its dated coefficient series has
positive same-date correlation with SPY. In a table: report the five leading directions
coefficients, the first-component squared-energy fraction, and the smallest $K$ whose cumulative
fraction is at least 80%. Plot the three leading directions side by side and the three
cumulative-fraction curves together. If there is a difference between the Sheet 3 and Sheet 4
results, are they caused by "SVD versus PCA," or by the preprocessing choice?

In [ ]:
def leading_direction(M):
    """Leading left singular vector of M (oriented so that its dated coefficient
    series sigma_1 v_1 is positively correlated with SPY) and cumulative energy fractions."""
    U_M, s_M, Vt_M = np.linalg.svd(M, full_matrices=False)
    u1, coeffs = U_M[:, 0], s_M[0] * Vt_M[0]
    if np.corrcoef(coeffs, R[0])[0, 1] < 0:   # R[0] = SPY
        u1 = -u1
    cumulative = np.cumsum(s_M**2) / np.sum(s_M**2)
    return u1, cumulative


analyses = {"raw SVD (R)": R, "covariance PCA (X)": X, "correlation PCA (Z)": Z}

table = {}
cumulative_curves = {}
for name, M in analyses.items():
    u1, cumulative = leading_direction(M)
    cumulative_curves[name] = cumulative
    table[name] = {**dict(zip(ASSETS, u1)),
                   "first-component fraction": cumulative[0],
                   "smallest K with >= 80%": int(np.argmax(cumulative >= 0.80)) + 1}

comparison = pd.DataFrame.from_dict(table, orient="index")    # one row per analysis
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, name in zip(axes, analyses):
    ax.bar(ASSETS, comparison.loc[name, ASSETS])
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"leading direction: {name}")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, cumulative in cumulative_curves.items():
    ax.plot(ks, cumulative, "o-", label=name)
ax.axhline(0.8, color="gray", ls="--")
ax.set_xticks(ks)
ax.set_xlabel("K")
ax.set_ylabel("cumulative fraction")
ax.set_title("Cumulative squared-energy / variance fractions")
ax.legend()
plt.show()

**Discussion.**